 # Tentativo Simple CNN 

 -Inserire descrizione di cosa faremo in questo notebook-

## Import

In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt


## Caricamento Dataset Split

In [5]:
# Assicuriamoci di usare lo stesso CSV appena creato
OUTPUT_CSV = '../train_val_split.csv'


# Ricarichiamo i dataframe dal CSV
try:
    df = pd.read_csv(OUTPUT_CSV)
    train_df = df[df['split'] == 'train']
    val_df = df[df['split'] == 'val']
    print(f"CSV caricato con successo. Train set: {len(train_df)}, Val set: {len(val_df)}")
except FileNotFoundError:
    raise FileNotFoundError(f"ERRORE: Non trovo il file {OUTPUT_CSV}. Sicuro che la FASE 1 sia andata a buon fine?")

CSV caricato con successo. Train set: 12412, Val set: 3103


## Resize Immagini 128 x 128

In [6]:
print("\n--- FASE 2: Caricamento tensori ---")

# Trasformazioni base: ridimensionamento a 128x128 per velocità  
base_transforms = transforms.Compose([
    transforms.Resize((128, 128)),  #cambia a 224 x 224 per secondo esperimento
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class WasteDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self): 
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.dataframe.iloc[idx]['filepath']
        image = Image.open(img_path).convert('RGB')
        label = self.dataframe.iloc[idx]['label']
        if self.transform: image = self.transform(image)
        return image, label

BATCH_SIZE = 16 
train_loader = DataLoader(WasteDataset(train_df, transform=base_transforms), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(WasteDataset(val_df, transform=base_transforms), batch_size=BATCH_SIZE, shuffle=False)
print("Immagini in memoria e pronte per la rete")


--- FASE 2: Caricamento tensori ---
Immagini in memoria e pronte per la rete


## Costruzione CNN e addestramento

In [7]:
print("\n--- FASE 3: Architettura Rete ---")

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)

        # 64 canali * 16 * 16 (dimensione spaziale residua partendo da 128 x 128)
        self.fc1 = nn.Linear(64 * 16 * 16, 128) #quando passi a 224 x 224 , camnbia in self.fc1 = nn.Linear(64 * 28 * 28, 128)
        self.fc2 = nn.Linear(128, 8) 

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = F.max_pool2d(F.relu(self.conv3(x)), 2)

        x = x.view(-1, 64 * 16 * 16) # quando passi a 224 x 224 , cambia in x = x.view(-1, 64 * 28 * 28
        x = F.relu(self.fc1(x))
        return self.fc2(x)

# Cerca prima una GPU NVIDIA (Windows/Linux), poi una GPU Mac, altrimenti usa la CPU
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
model = SimpleCNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"Modello creato. Acceleratore in uso : {device.type.upper()}")


--- FASE 3: Architettura Rete ---
Modello creato. Acceleratore in uso : MPS


## Addestramento SimpleCNN pura


In [10]:
# --- Cella 4: Addestramento SimpleCNN  ---

print("\n--- FASE 0: Addestramento SimpleCNN---")

EPOCHS = 10                  # Lasciamo 10, ma l'Early Stopping la fermerà prima se diverge
PATIENCE = 2                 # Tolleranza di peggioramento sulla Validation Loss
best_val_loss = float('inf') # Memoria del minimo storico della Loss
patience_counter = 0

# Rinominato per la Fase 0
BEST_MODEL_PATH = 'simplecnn_weights.pth'

# Dizionari per salvare la history e plotttare il grafico dell'overfitting per il report
history = {'train_loss': [], 'val_loss': [], 'bal_acc': []}

for epoch in range(EPOCHS):
    
    # --- FASE DI TRAINING ---
    model.train()
    running_loss = 0.0
    total_train_loss = 0.0

    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        total_train_loss += loss.item()

        if i % 20 == 19: 
            print(f"  Epoca [{epoch+1}/{EPOCHS}], Batch [{i+1}/{len(train_loader)}], Loss: {running_loss/20:.4f}")
            running_loss = 0.0

    print(f" Epoca {epoch+1} conclusa. Calcolo metriche di validazione...")

    # --- FASE DI VALIDATION ---
    model.eval()
    all_preds = []
    all_labels = []
    total_val_loss = 0.0 

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)

            val_loss = criterion(outputs, labels)
            total_val_loss += val_loss.item()

            _, predicted = torch.max(outputs.data, 1)
            all_preds.append(predicted.cpu())
            all_labels.append(labels.cpu())  # CORRETTO: Spostato su CPU prima di Numpy!
            
    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    # Calcolo medie di fine epoca
    avg_train_loss = total_train_loss / len(train_loader)
    avg_val_loss = total_val_loss / len(val_loader)
    bal_acc = balanced_accuracy_score(all_labels, all_preds) * 100
            
    # Salvataggio history
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['bal_acc'].append(bal_acc)

    print(f"--> Riepilogo Epoca {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Balanced Accuracy: {bal_acc:.2f}%\n")
    
    # --- LOGICA DI EARLY STOPPING E SALVATAGGIO ---
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        best_preds_memory = all_preds.copy()
        print(f"Val Loss migliorata! Checkpoint ottimale salvato su '{BEST_MODEL_PATH}'\n")
    else:
        patience_counter += 1
        print(f"Val Loss peggiorata o stabile. (Patience: {patience_counter}/{PATIENCE})\n")
        
        if patience_counter >= PATIENCE:
            print(f"EARLY STOPPING INNESCATO ALL'EPOCA {epoch+1}! Interruzione per prevenire Overfitting.")
            break

print("Addestramento concluso!")


--- FASE 0: Addestramento SimpleCNN---


FileNotFoundError: [Errno 2] No such file or directory: '../dataset/waste_type_identification/clothing/clothes/clothes1822.jpg'

## Metriche di Addestramento e Confusion Matrix

In [ ]:
########## DA ESEGUIRE PER LE METRICHE DI UN ADDESTRAMENTO ##############

# Nomi delle classi ordinati dal dizionario class_mapping (0 fino a 7)
class_names = ['Battery', 'Clothing', 'Glass', 'Metal', 'Organic', 'Papery', 'Plastic', 'Undifferentiated']

# 1. Report completo accademico (Precision, Recall, F1-Score per ogni singola classe)
print("\nCLASSIFICATION REPORT :")
print(classification_report(all_labels, best_preds_memory, target_names=class_names))

# 2. Matrice di Confusione
cm = confusion_matrix(all_labels, best_preds_memory)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title("Matrice di Confusione - SimpleCNN pura")
plt.ylabel("Classe Reale (Ground Truth)")
plt.xlabel("Classe Preditta dal Modello")
plt.show()

## Analisi Immagini


In [ ]:


################# ANALISI IMMAGINI ##################

def contrastive_error_forensics(model, val_loader, device, class_names, true_class, confused_class, samples_per_row=5):
    """
    Confronta visivamente i campioni falliti con i campioni perfetti della stessa classe.
    """
    model.eval()
    true_idx = class_names.index(true_class)
    conf_idx = class_names.index(confused_class)
    
    mistakes_img, mistakes_conf = [], []
    corrects_img, corrects_conf = [], []
    
    # Costanti di denormalizzazione ImageNet
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs_dev = inputs.to(device)
            outputs = model(inputs_dev)
            probs = F.softmax(outputs, dim=1)
            confs, preds = torch.max(probs, 1)
            
            for i in range(len(labels)):
                lbl = labels[i].item()
                prd = preds[i].item()
                cnf = confs[i].item() * 100
                
                # Caso A: Errore mirato (True Class -> Confused Class)
                if lbl == true_idx and prd == conf_idx and len(mistakes_img) < samples_per_row:
                    img = torch.clamp(inputs[i].cpu() * std + mean, 0, 1).permute(1, 2, 0).numpy()
                    mistakes_img.append(img)
                    mistakes_conf.append(cnf)
                
                # Caso B: Benchmark di perfezione (True Class -> True Class con altissima sicurezza)
                elif lbl == true_idx and prd == true_idx and cnf > 92.0 and len(corrects_img) < samples_per_row:
                    img = torch.clamp(inputs[i].cpu() * std + mean, 0, 1).permute(1, 2, 0).numpy()
                    corrects_img.append(img)
                    corrects_conf.append(cnf)
                    
                if len(mistakes_img) == samples_per_row and len(corrects_img) == samples_per_row:
                    break
            if len(mistakes_img) == samples_per_row and len(corrects_img) == samples_per_row:
                break

    # Creazione della tavola ottica comparativa
    fig, axes = plt.subplots(2, samples_per_row, figsize=(16, 6.5))
    fig.suptitle(f"ANALISI CONTRASTIVA DI CLASSE: [{true_class.upper()}]", fontsize=14, fontweight='bold', y=0.98)
    
    # Render Riga 1: Gli Errori
    for idx in range(samples_per_row):
        ax = axes[0, idx]
        if idx < len(mistakes_img):
            ax.imshow(mistakes_img[idx])
            ax.set_title(f"Scambiato per: {confused_class}\nSicurezza Errore: {mistakes_conf[idx]:.1f}%", color='darkred', fontweight='bold', fontsize=10)
        else:
            ax.text(0.5, 0.5, "Nessun altro\nerrore trovato", ha='center', va='center')
            
        ax.axis('off')
        if idx == 0: ax.set_ylabel("FALLIMENTI", fontsize=12, fontweight='bold', color='red')

    # Render Riga 2: I Successi
    for idx in range(samples_per_row):
        ax = axes[1, idx]
        if idx < len(corrects_img):
            ax.imshow(corrects_img[idx])
            ax.set_title(f"Predizione Corretta\nSicurezza Rete: {corrects_conf[idx]:.1f}%", color='darkgreen', fontweight='bold', fontsize=10)
        else:
            ax.text(0.5, 0.5, "Campionamento\ninsufficiente", ha='center', va='center')
            
        ax.axis('off')
        if idx == 0: ax.set_ylabel("BENCHMARK", fontsize=12, fontweight='bold', color='green')

    plt.tight_layout()
    plt.show()

# --- ESEGUI IL TEST CONTRASTIVO ---
# Sostituisci 'Plastic' e 'Glass' con la coppia critica che vedi spuntare dalla tua Matrice di Confusione!
class_names = ['Battery', 'Clothing', 'Glass', 'Metal', 'Organic', 'Papery', 'Plastic', 'Undifferentiated']

contrastive_error_forensics(model, val_loader, device, class_names, true_class='Plastic', confused_class='Glass')